## Experiment Ariadne

Aim is to correlate OOD-deltas with ID-deltas. For this, we'll collect Hard negative examples from the rollouts per checkpoints - questions which contain a calculation error. To filter eligible traces, we'll look for subsets of traces which have non-zero-solverate, s.t. we obtain ID-gts and ID-negative samples.

Later on, we'll pipe the negative samples through a judge model to decide whether there is a calculation error present which lead to the wrong result. For those samples, we'll create tuples $(gt_{ID}, aug_{ID})$ which we use to measure the in-distribution delta between making a calculation error and correctly solving the task.

#### 09.12 Update
- removing negative samples which contain "```python" keyword, as python-code hallucination seems to be prevalent in Qwen2.5-7B generated answers.
- removing answers which aren't parsable, i.e. which do not contain "\\boxed{}" in last 300 characters

#### 10.12 Update
- making the prompt way simpler -> removing examples, making prompt slim

#### 17.12 Update
- Also aim for a reasoning strategy.

In [1]:
import os
import pickle
import pandas as pd
from utils import load_rollouts, sample


In [2]:
RELOAD=False
if RELOAD:
    rollouts : list[pd.DataFrame] = load_rollouts()
    with open('rollouts.parquet', 'wb') as file:
        pickle.dump(obj=rollouts, file=file)
else:
    with open('rollouts.parquet', 'rb') as file:
        rollouts = pickle.load(file)

In [6]:
# For calculation errors ID samples, we require to look at questions which do have negative AND positive responses.
negatives : list[pd.DataFrame] = [
    sample(df, sample_qs=-1, only_filter='negative', filter_std_nonzero=True, filter_overlong=False) for df in rollouts]

In [7]:
list(map(len, negatives))

[971, 1064, 908, 735, 676, 615, 604, 587, 527]

In [8]:
# We have to construct the judge prompt from input, output and gts.
def classify_calc_error_prompt(row : pd.DataFrame) -> list[dict]:
    system = "You are a helpful judge and an expert in mathematical reasoning."
    
    prefix = (
        "You're given a question and an incorrect students answer. Answer '#### yes' if and only if the wrong answer "
        "is caused by an arithmetic error or an algebraic error, i.e. an error involving simplification, summation, multiplication etc. "
        "Answer '#### no' if the wrong answer is caused by another error type e.g. a reasoning error, logic error or false assumptions.\n"
    )
    remove_prefix, remove_suffix = "system\nYou are a helpful assistant.\nuser\n", " Let's think step by step and output the final answer within \\boxed{}.\nassistant\n"
    question = row['input'].removeprefix(remove_prefix).removesuffix(remove_suffix)
    student = row['output']
    gt = row['gts']
    prompt = [{
        'role' : 'system',
        'content' : system
    },
    {
        'role' : 'user',
        'content' : f"{prefix}\n\nQuestion:\n\n{question}\n Student's incorrect answer:\n\n{student}\nGround Truth Solution:\n\n{gt}\n Please only answer either '#### yes' or '#### no'.\n"
    }]
    return prompt

In [ ]:
pdfs = [pd.DataFrame({
    'prompt' : df.apply(classify_calc_error_prompt, axis=1),
    'step' : df['step'],
    'old_index' : df.index
}) for df in negatives]

In [16]:
df = pd.concat(pdfs, axis=0)
df = df.reset_index(drop=True)
df.head(2)

,prompt,step,old_index
0,"[{'role': 'system', 'content': 'You are a help...",80,364
1,"[{'role': 'system', 'content': 'You are a help...",80,2431


In [17]:
os.makedirs('/u/rfechner/data/ariadne', exist_ok=True)
with open('/u/rfechner/data/ariadne/id-prompts-simple.parquet', 'wb') as file:
    df.to_parquet(file)

### Verification Classification

In [12]:
# Validation ID samples. No filtering for std_nonzero, overlong sequences etc needed.
positives : list[pd.DataFrame] = [
    sample(df, sample_qs=-1, only_filter='positive', filter_std_nonzero=False, filter_overlong=False) for df in rollouts]

In [13]:
list(map(len, positives))

[1296, 1353, 1359, 1365, 1359, 1338, 1335, 1323, 1314]

In [12]:
def construct_prompt_verify(row : pd.DataFrame) -> list[dict]:
    system = "You are a helpful judge and an expert in mathematical reasoning."
    
    prefix = (
        "You're given a question and a correct students answer. Answer '#### yes' if and only if the answer "
        "contains explicit (verbalized) verification of any intermediate or the final result. This includes explicit re-iterating over answers "
        "to check their correctness, verification by plugging in the found result back into the equation to check the answer satisfies "
        "conditions etc. This may include student answers which contain 'Let's verify the answer' or 'Let's check the answer' etc."
        "\nHowever, this does not include step-by-step initial solution of the problem, we're only looking for verbalized verification after "
        "a solution was already computed. "
        "Answer '#### no' if the answer doesn't contain verification.\n"
    )
    remove_prefix, remove_suffix = "system\nYou are a helpful assistant.\nuser\n", " Let's think step by step and output the final answer within \\boxed{}.\nassistant\n"
    question = row['input'].removeprefix(remove_prefix).removesuffix(remove_suffix)
    student = row['output']
    gt = row['gts']
    prompt = [{
        'role' : 'system',
        'content' : system
    },
    {
        'role' : 'user',
        'content' : f"{prefix}\n\nQuestion:\n\n{question}\n\nStudent's correct answer:\n\n{student}\n\nGround Truth Solution:\n\n{gt}\n\nPlease only answer either '#### yes' or '#### no'.\n"
    }]
    return prompt

In [ ]:
out_verify = pd.concat([pd.DataFrame({
    'prompt' : df.apply(construct_prompt_verify, axis=1),
    'step' : df['step'],
    'old_index' : df.index
}) for df in positives])

In [15]:
os.makedirs('/u/rfechner/data/ariadne', exist_ok=True)
with open('/u/rfechner/data/ariadne/id-prompts-verify.parquet', 'wb') as file:
    out_verify.to_parquet(file)